# LangGraph example: a three-layer number pipeline

Type a number. One agent checks it is a whole number from 1 to 10, three agents each own a range, three more add facts, and a last one confirms the number. It is shaped like the incident-triage graph, with no Azure and no LLM.

**To run it:** Python 3.12 and `pip install langgraph`.

## The LangGraph pieces you will meet

Each cell below tags the pieces it uses, so you can spot them.

| Tag | What it is | Plain meaning | In this notebook |
|---|---|---|---|
| 🟦 **STATE** | A shared dict (`TypedDict`) every node can read | The whiteboard | `Pipeline` |
| 🟩 **NODE** | A plain function: reads state, returns only what it changes | One worker | `validate`, `low`, `mid`, ... |
| ➡️ **EDGE** | "After A, run B" | An arrow | `add_edge("reject", END)` |
| 🔀 **CONDITIONAL EDGE** | A function picks the next node(s) | A decision diamond | after `validate` |
| 🔱 **FAN-OUT** | One node leads to several, which run at the same time | Hand out jobs | layer 1 to layer 2 |
| 🔗 **FAN-IN** | A node with a list of sources waits for all of them | Wait for everyone | layer 2 to layer 3 |
| 🧮 **REDUCER** | A rule for merging parallel writes to one key | Merge the notes | `merge_findings`, `operator.add` |
| 🏁 **START / END** | The entry and exit of the graph | Door in, door out | `START`, `END` |

## The graph

```
 START
   |
   v
 LAYER 1   [ validate ] -- bad input --> [ reject ] --> END
   |
   | valid number: start all three at once
   v
 LAYER 2   [ low 1-3 ]     [ mid 4-7 ]     [ high 8-10 ]
                 \              |              /
                  \   wait until ALL THREE finish
                   \            |            /
                    v           v           v
 LAYER 3   [ even_odd ]      [ prime ]      [ size ]
                 \              |              /
                  \   wait until ALL THREE finish
                   \            |            /
                    v           v           v
 LAYER 4                   [ finalize ]
                                |
                                v
                               END
```

Read it top to bottom. Agents on the same row run at the same time. A new row starts only when every agent in the row above has finished.

| Layer | Nodes | Like in the project | LangGraph pieces |
|---|---|---|---|
| 1 | `validate`, `reject` | Triage supervisor | 🔀 conditional edge, 🔱 fan-out |
| 2 | `low`, `mid`, `high` | Logs, Metrics, Change agents | 🟩 nodes, 🧮 reducer, 🔗 fan-in |
| 3 | `even_odd`, `prime`, `size` | a second wave of specialists | 🔱 fan-out again, 🔗 fan-in |
| 4 | `finalize` | Postmortem | 🔗 fan-in, ➡️ edge to END |

**How this notebook is ordered:** one cell per node, in the order LangGraph executes them. A node is a plain function, so each cell runs it by hand on a running `state` and prints what it returned. Near the end, one cell wires the real graph and runs it, to prove it gives the same answer.

Change `RAW` in the first code cell (try `6`, `2`, `11`, `abc`) and run again from the top to follow another path.

## Step 0: the state, the reducers and the input

**What happens here:** we define the 🟦 **STATE**, the dict that travels through the whole graph, and put the user's input in it. Every node reads the state and returns only the keys it wants to change.

| Key | Written by | Reducer | Effect |
|---|---|---|---|
| `findings` | layer 2 and layer 3 nodes | `merge_findings` (merge two dicts) | all their facts end up in one dict |
| `trace` | every node | `operator.add` (join two lists) | one log line per node, in order |

`apply(state, update)` plays LangGraph's role in this notebook: it merges a node's return into the state, using the same reducer rules. `say()` builds one `trace` line, like `em.say` in the project.

🏁 The graph starts here: `START` hands `{"raw": RAW}` to the first node.

In [1]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph

RAW = "6"          # <- the user's input. Try "2", "11", "abc" and run again from the top.


def merge_findings(a: dict, b: dict) -> dict:       # 🧮 REDUCER: how to combine two writes
    return {**a, **b}


print(merge_findings({"low": False}, {"mid": True}))   # first row of the table below


{'low': False, 'mid': True}


### What `merge_findings` takes and returns

It takes two dicts and returns one dict with the keys of both. If a key is in both, the second one (`b`) wins.

| `a` (already in the state) | `b` (new write) | Output |
|---|---|---|
| `{"low": False}` | `{"mid": True}` | `{"low": False, "mid": True}` |
| `{"low": False, "mid": True}` | `{"high": False}` | `{"low": False, "mid": True, "high": False}` |
| `{}` | `{"parity": "even"}` | `{"parity": "even"}` |
| `{"mid": True}` | `{"mid": False}` | `{"mid": False}` (same key: `b` wins) |

LangGraph calls it for you whenever two nodes write `findings`. With three parallel nodes it is called once per write, so the final dict holds all three answers.

In [2]:
class Pipeline(TypedDict, total=False):             # 🟦 STATE: the shared whiteboard
    raw: str                                              # what the user typed
    number: int                                           # set by validate
    error: str                                            # set by validate when the input is bad
    findings: Annotated[dict, merge_findings]             # layers 2 and 3 all write here
    trace: Annotated[list, operator.add]                  # every node appends a line
    final: int                                            # set by finalize


def say(node: str, text: str) -> dict:
    return {"trace": [f"{node:9} {text}"]}


def apply(state: dict, update: dict) -> dict:       # what LangGraph does after each node
    for key, value in update.items():
        if key == "findings":
            state[key] = merge_findings(state.get(key, {}), value)
        elif key == "trace":
            state[key] = state.get(key, []) + value
        else:
            state[key] = value
    return state


state = {"raw": RAW}                                # 🏁 START: the graph begins with only the input
print("state at START:", state)

state at START: {'raw': '6'}


### What `say` and `apply` take and return

| Function | Input | Output |
|---|---|---|
| `say` | `"validate"`, `"hello"` | `{"trace": ["validate  hello"]}` (a one-line log, ready to be a node's return) |
| `apply` | state `{"raw": "6"}`, update `{"number": 6}` | state `{"raw": "6", "number": 6}` (new key added) |
| `apply` | state `{"findings": {"low": False}}`, update `{"findings": {"mid": True}}` | `findings` becomes `{"low": False, "mid": True}` (merged by `merge_findings`) |
| `apply` | state `{"trace": ["a"]}`, update `{"trace": ["b"]}` | `trace` becomes `["a", "b"]` (lists joined) |
| `apply` | state `{"number": 6}`, update `{"number": 7}` | `number` becomes `7` (any other key is simply replaced) |

## 1. `validate` (layer 1, first node)

**What happens here:** the edge `START -> validate` runs this node first. It checks the input is a whole number from 1 to 10.

| Piece | Name | One line: what it does |
|---|---|---|
| 🟩 **NODE** | `validate` | Check the input. Save it as `number`, or save an `error`. |

It returns **only** its own keys (`number` or `error`, and a `trace` line). The cell then merges that into `state`.

In [3]:
LAYER2 = ["low", "mid", "high"]          # names of the layer 2 nodes, used by the router and the wiring
LAYER3 = ["even_odd", "prime", "size"]   # names of the layer 3 nodes, used in the wiring


def validate(state: Pipeline) -> dict:                       # 🟩 NODE
    """Check the input is a whole number from 1 to 10. Save it as `number`, or save an `error`."""
    raw = state["raw"].strip()
    if not raw.lstrip("-").isdigit():
        return {"error": f"'{raw}' is not a whole number", **say("validate", "rejected: not a number")}
    n = int(raw)
    if not 1 <= n <= 10:
        return {"error": f"{n} is outside 1-10", **say("validate", "rejected: out of range")}
    return {"number": n, **say("validate", f"{n} is valid, sending to the range agents")}


update = validate(state)
print("validate returned:", update)
state = apply(state, update)
print("state now:", {k: v for k, v in state.items() if k != "trace"})

validate returned: {'number': 6, 'trace': ['validate  6 is valid, sending to the range agents']}
state now: {'raw': '6', 'number': 6}


### What `validate` takes and returns

Input is the state (only `raw` is read). Output is only the keys it changes.

| `raw` | Output |
|---|---|
| `"6"` | `{"number": 6, "trace": ["validate  6 is valid, ..."]}` |
| `" 4 "` | `{"number": 4, ...}` (spaces are stripped) |
| `"10"` | `{"number": 10, ...}` (10 is allowed) |
| `"11"` | `{"error": "11 is outside 1-10", "trace": ["validate  rejected: out of range"]}` |
| `"-3"` | `{"error": "-3 is outside 1-10", ...}` |
| `"abc"` | `{"error": "'abc' is not a whole number", "trace": ["validate  rejected: not a number"]}` |
| `"2.5"` | `{"error": "'2.5' is not a whole number", ...}` |

## 2. The router (conditional edge after `validate`)

**What happens here:** after `validate`, LangGraph does not follow a fixed arrow. It calls a small function and uses the answer to pick the next node(s).

| Piece | Name | One line: what it does |
|---|---|---|
| 🔀 **CONDITIONAL EDGE** | `route_after_validate` | Returns `"reject"` if there is an error, otherwise the **list** `["low", "mid", "high"]`. |
| 🔱 **FAN-OUT** | the list | A list means: start all of these at the same time. |

In [4]:
def route_after_validate(state: Pipeline):                   # 🔀 CONDITIONAL EDGE (router)
    """Decide what runs next: `reject` if there is an error, otherwise all of layer 2 at once."""
    return "reject" if state.get("error") else LAYER2        # a list = run all of these in parallel


print("router sends the run to:", route_after_validate(state))

router sends the run to: ['low', 'mid', 'high']


### What `route_after_validate` takes and returns

It reads the state, changes nothing, and returns the name(s) of the next node(s).

| State | Output | Meaning |
|---|---|---|
| `{"number": 6, ...}` (no `error`) | `["low", "mid", "high"]` | a list: start all three at once |
| `{"error": "11 is outside 1-10", ...}` | `"reject"` | one node: go to `reject` |

## 3. `low` (layer 2, first of three parallel nodes)

**What happens here:** the router started `low`, `mid` and `high` together. In this notebook we run them one after another, but in LangGraph they run **at the same time** and each reads the same state.

| Piece | Name | One line: what it does |
|---|---|---|
| 🟩 **NODE** (built by a factory) | `low` | Is `number` in 1-3? Writes `findings["low"]` = True or False. |
| 🧮 **REDUCER** | `merge_findings` | The three agents write to `findings` together; the reducer merges their small dicts. |

`range_agent(name, low, high)` is a factory: one function that builds all three nodes.

*If you chose a bad input (11 or abc), the router sent the run to `reject` instead. Skip to the cell "The other path: `reject`".*

In [5]:
def range_agent(name: str, low: int, high: int):         # a factory that builds one 🟩 NODE
    def node(state: Pipeline) -> dict:
        mine = low <= state["number"] <= high
        text = "my range, claiming it" if mine else "not mine"
        return {"findings": {name: mine}, **say(name, f"{low}-{high}: {text}")}
    node.__name__ = name
    return node


low_node = range_agent("low", 1, 3)

update = low_node(state)
print("low returned:", update)
state = apply(state, update)
print("state now:", {k: v for k, v in state.items() if k != "trace"})

low returned: {'findings': {'low': False}, 'trace': ['low       1-3: not mine']}
state now: {'raw': '6', 'number': 6, 'findings': {'low': False}}


### What `range_agent` and `low` take and return

`range_agent(name, low, high)` takes a name and a range and **returns a node function**. That node reads `number` and returns only `findings` and `trace`.

| Call | Input state | Output |
|---|---|---|
| `range_agent("low", 1, 3)` | (none, it builds a function) | a node named `low` |
| `low_node(...)` | `{"number": 2}` | `{"findings": {"low": True}, "trace": ["low       1-3: my range, claiming it"]}` |
| `low_node(...)` | `{"number": 6}` | `{"findings": {"low": False}, "trace": ["low       1-3: not mine"]}` |

## 4. `mid` (layer 2, parallel with `low`)

**What happens here:** the same kind of node, for the range 4-7. It does not look at what `low` wrote; it only reads `number`. That is why these three can safely run in parallel.

| Piece | Name | One line: what it does |
|---|---|---|
| 🟩 **NODE** | `mid` | Is `number` in 4-7? Writes `findings["mid"]`. |

In [6]:
mid_node = range_agent("mid", 4, 7)

update = mid_node(state)
print("mid returned:", update)
state = apply(state, update)
print("state now:", {k: v for k, v in state.items() if k != "trace"})

mid returned: {'findings': {'mid': True}, 'trace': ['mid       4-7: my range, claiming it']}
state now: {'raw': '6', 'number': 6, 'findings': {'low': False, 'mid': True}}


### What `mid` takes and returns

| Input state | Output |
|---|---|
| `{"number": 6}` | `{"findings": {"mid": True}, "trace": ["mid       4-7: my range, claiming it"]}` |
| `{"number": 2}` | `{"findings": {"mid": False}, "trace": ["mid       4-7: not mine"]}` |

## 5. `high` (layer 2, parallel with `low` and `mid`)

**What happens here:** the third range agent, for 8-10. After this one, the whole of layer 2 has finished.

| Piece | Name | One line: what it does |
|---|---|---|
| 🟩 **NODE** | `high` | Is `number` in 8-10? Writes `findings["high"]`. |
| 🧮 **REDUCER** | `merge_findings` | `findings` now holds all three answers. |

In [7]:
high_node = range_agent("high", 8, 10)

update = high_node(state)
print("high returned:", update)
state = apply(state, update)
print("state now:", {k: v for k, v in state.items() if k != "trace"})

high returned: {'findings': {'high': False}, 'trace': ['high      8-10: not mine']}
state now: {'raw': '6', 'number': 6, 'findings': {'low': False, 'mid': True, 'high': False}}


### What `high` takes and returns

| Input state | Output |
|---|---|
| `{"number": 9}` | `{"findings": {"high": True}, "trace": ["high      8-10: my range, claiming it"]}` |
| `{"number": 6}` | `{"findings": {"high": False}, "trace": ["high      8-10: not mine"]}` |

## 6. `even_odd` (layer 3, first of three parallel nodes)

**What happens here:** layer 3 starts only after **all three** layer 2 nodes have finished. That wait is a 🔗 **FAN-IN**: in the wiring, the source of the edge is the list `["low", "mid", "high"]`. Then the three layer 3 nodes start together (🔱 **FAN-OUT** again).

| Piece | Name | One line: what it does |
|---|---|---|
| 🟩 **NODE** | `even_odd` | Writes `findings["parity"]`: even or odd. |

In [8]:
def even_odd(state: Pipeline) -> dict:                   # 🟩 NODE
    """Say whether the number is even or odd."""
    kind = "even" if state["number"] % 2 == 0 else "odd"
    return {"findings": {"parity": kind}, **say("even_odd", kind)}


update = even_odd(state)
print("even_odd returned:", update)
state = apply(state, update)
print("state now:", {k: v for k, v in state.items() if k != "trace"})

even_odd returned: {'findings': {'parity': 'even'}, 'trace': ['even_odd  even']}
state now: {'raw': '6', 'number': 6, 'findings': {'low': False, 'mid': True, 'high': False, 'parity': 'even'}}


### What `even_odd` takes and returns

| Input state | Output |
|---|---|
| `{"number": 6}` | `{"findings": {"parity": "even"}, "trace": ["even_odd  even"]}` |
| `{"number": 7}` | `{"findings": {"parity": "odd"}, "trace": ["even_odd  odd"]}` |

## 7. `prime` (layer 3, parallel with `even_odd`)

| Piece | Name | One line: what it does |
|---|---|---|
| 🟩 **NODE** | `prime` | Writes `findings["prime"]`: True or False. |

In [9]:
def prime(state: Pipeline) -> dict:                      # 🟩 NODE
    """Say whether the number is prime."""
    n = state["number"]
    is_prime = n > 1 and all(n % d for d in range(2, n))
    return {"findings": {"prime": is_prime}, **say("prime", "prime" if is_prime else "not prime")}


update = prime(state)
print("prime returned:", update)
state = apply(state, update)
print("state now:", {k: v for k, v in state.items() if k != "trace"})

prime returned: {'findings': {'prime': False}, 'trace': ['prime     not prime']}
state now: {'raw': '6', 'number': 6, 'findings': {'low': False, 'mid': True, 'high': False, 'parity': 'even', 'prime': False}}


### What `prime` takes and returns

| Input state | Output |
|---|---|
| `{"number": 7}` | `{"findings": {"prime": True}, "trace": ["prime     prime"]}` |
| `{"number": 6}` | `{"findings": {"prime": False}, "trace": ["prime     not prime"]}` |
| `{"number": 1}` | `{"findings": {"prime": False}, ...}` (1 is not prime) |

## 8. `size` (layer 3, parallel with `even_odd` and `prime`)

**What happens here:** the last layer 3 node. When it finishes, layer 3 is done.

| Piece | Name | One line: what it does |
|---|---|---|
| 🟩 **NODE** | `size` | Writes `findings["size"]`: small (1-3), medium (4-7) or large (8-10). |

In [10]:
def size(state: Pipeline) -> dict:                       # 🟩 NODE
    """Label the number small, medium or large."""
    n = state["number"]
    label = "small" if n <= 3 else "medium" if n <= 7 else "large"
    return {"findings": {"size": label}, **say("size", label)}


update = size(state)
print("size returned:", update)
state = apply(state, update)
print("state now:", {k: v for k, v in state.items() if k != "trace"})

size returned: {'findings': {'size': 'medium'}, 'trace': ['size      medium']}
state now: {'raw': '6', 'number': 6, 'findings': {'low': False, 'mid': True, 'high': False, 'parity': 'even', 'prime': False, 'size': 'medium'}}


### What `size` takes and returns

| `number` | Output `findings` |
|---|---|
| 1, 2, 3 | `{"size": "small"}` |
| 4, 5, 6, 7 | `{"size": "medium"}` |
| 8, 9, 10 | `{"size": "large"}` |

## 9. `finalize` (layer 4, last node)

**What happens here:** `finalize` waits for all of layer 3 (🔗 **FAN-IN**), reads everything in `findings`, and writes the answer to `final`. Then the edge `finalize -> END` stops the graph.

| Piece | Name | One line: what it does |
|---|---|---|
| 🟩 **NODE** | `finalize` | Find which range agent claimed the number, write `final`. |
| ➡️ **EDGE** + 🏁 **END** | `finalize -> END` | The graph is finished. |

In [11]:
def finalize(state: Pipeline) -> dict:                   # 🟩 NODE
    """Confirm the number: find which range agent claimed it and save it as `final`."""
    owner = next(k for k in ("low", "mid", "high") if state["findings"][k])
    return {"final": state["number"], **say("finalize", f"{state['number']} confirmed by '{owner}'")}


update = finalize(state)
print("finalize returned:", update)
state = apply(state, update)
print("\nTRACE (the log every node added):")
for line in state["trace"]:
    print("   ", line)
print("\nfinal answer:", state["final"])
manual_result = state

finalize returned: {'final': 6, 'trace': ["finalize  6 confirmed by 'mid'"]}

TRACE (the log every node added):
    validate  6 is valid, sending to the range agents
    low       1-3: not mine
    mid       4-7: my range, claiming it
    high      8-10: not mine
    even_odd  even
    prime     not prime
    size      medium
    finalize  6 confirmed by 'mid'

final answer: 6


### What `finalize` takes and returns

It reads `number` and `findings`, finds the range agent whose answer is `True`, and returns only `final` and `trace`.

| Input state | Output |
|---|---|
| `number` 6, `findings` has `"mid": True` | `{"final": 6, "trace": ["finalize  6 confirmed by 'mid'"]}` |
| `number` 2, `findings` has `"low": True` | `{"final": 2, "trace": ["finalize  2 confirmed by 'low'"]}` |

## The other path: `reject`

**What happens here:** only for bad input. The router returns `"reject"` instead of the layer 2 list, so layers 2 to 4 never run. The edge `reject -> END` stops the graph.

| Piece | Name | One line: what it does |
|---|---|---|
| 🟩 **NODE** | `reject` | Log why the input was refused. |
| ➡️ **EDGE** + 🏁 **END** | `reject -> END` | The second way the graph can finish. |

The cell runs this path on its own with two bad inputs, so you can see it without changing `RAW`.

In [12]:
def reject(state: Pipeline) -> dict:                         # 🟩 NODE
    """Log why the input was refused. The graph then ends."""
    return say("reject", state["error"])


for bad in ("11", "abc"):
    s = apply({"raw": bad}, validate({"raw": bad}))
    print(f"input {bad!r}: router sends it to {route_after_validate(s)!r}")
    s = apply(s, reject(s))
    print("   trace:", s["trace"], "\n")

input '11': router sends it to 'reject'
   trace: ['validate  rejected: out of range', 'reject    11 is outside 1-10'] 

input 'abc': router sends it to 'reject'
   trace: ['validate  rejected: not a number', "reject    'abc' is not a whole number"] 



### What `reject` takes and returns

| Input state | Output |
|---|---|
| `{"error": "11 is outside 1-10", ...}` | `{"trace": ["reject    11 is outside 1-10"]}` |
| `{"error": "'abc' is not a whole number", ...}` | `{"trace": ["reject    'abc' is not a whole number"]}` |

## Now the real graph: wiring and running

**What happens here:** until now we ran the functions by hand and merged the results ourselves. This cell hands the same functions to LangGraph, which does the ordering, the waiting and the merging for us.

| Call | Piece | Meaning |
|---|---|---|
| `StateGraph(Pipeline)` | 🟦 STATE | "A graph whose state looks like `Pipeline`" |
| `add_node(name, fn)` | 🟩 NODE | Register a function under a name |
| `add_edge(START, "validate")` | ➡️ EDGE + 🏁 START | The first node to run |
| `add_conditional_edges("validate", route_after_validate, [...])` | 🔀 CONDITIONAL EDGE | After `validate`, ask the router where to go |
| `add_edge(LAYER2, agent)` | 🔗 FAN-IN | A **list** as the source means: wait until all of them finish |
| `add_edge(LAYER3, "finalize")` | 🔗 FAN-IN | `finalize` waits for all of layer 3 |
| `add_edge("finalize", END)`, `add_edge("reject", END)` | ➡️ EDGE + 🏁 END | The two ways the graph can stop |
| `compile()` | | Check the wiring and make it runnable |

Then `pipeline.stream(..., stream_mode="updates")` runs it and prints each node's update **in the order LangGraph executed it**. Inside a layer the order may differ between runs, because parallel nodes have no fixed order. The last line checks the result equals the one we built by hand.

In [13]:
g = StateGraph(Pipeline)                                     # 🟦 STATE

g.add_node("validate", validate)                             # 🟩 NODES
g.add_node("reject", reject)
g.add_node("low", low_node)
g.add_node("mid", mid_node)
g.add_node("high", high_node)
for fn in (even_odd, prime, size, finalize):
    g.add_node(fn.__name__, fn)

g.add_edge(START, "validate")                                # ➡️ EDGE from 🏁 START
g.add_conditional_edges("validate", route_after_validate, ["reject", *LAYER2])   # 🔀 + 🔱
g.add_edge("reject", END)                                    # ➡️ EDGE to 🏁 END
for agent in LAYER3:
    g.add_edge(LAYER2, agent)                                # 🔗 FAN-IN, then 🔱 FAN-OUT
g.add_edge(LAYER3, "finalize")                               # 🔗 FAN-IN
g.add_edge("finalize", END)                                  # ➡️ EDGE to 🏁 END

pipeline = g.compile()

print("edges ('..>' = conditional):")
for edge in pipeline.get_graph().edges:
    arrow = "..>" if edge.conditional else "-->"
    print(f"  {edge.source:10} {arrow} {edge.target}")

print(f"\nrunning the real graph on {RAW!r}, node by node:")
for step in pipeline.stream({"raw": RAW}, stream_mode="updates"):
    for node_name, node_update in step.items():
        print(f"  {node_name:9} ->", node_update)

real = pipeline.invoke({"raw": RAW})
same = real.get("final") == manual_result.get("final") and real.get("findings") == manual_result.get("findings")
print("\nsame result as the cells above:", same)

edges ('..>' = conditional):
  __start__  --> validate
  even_odd   --> finalize
  high       --> even_odd
  high       --> prime
  high       --> size
  low        --> even_odd
  low        --> prime
  low        --> size
  mid        --> even_odd
  mid        --> prime
  mid        --> size
  prime      --> finalize
  size       --> finalize
  validate   ..> high
  validate   ..> low
  validate   ..> mid
  validate   ..> reject
  finalize   --> __end__
  reject     --> __end__

running the real graph on '6', node by node:
  validate  -> {'number': 6, 'trace': ['validate  6 is valid, sending to the range agents']}
  high      -> {'findings': {'high': False}, 'trace': ['high      8-10: not mine']}
  low       -> {'findings': {'low': False}, 'trace': ['low       1-3: not mine']}
  mid       -> {'findings': {'mid': True}, 'trace': ['mid       4-7: my range, claiming it']}
  even_odd  -> {'findings': {'parity': 'even'}, 'trace': ['even_odd  even']}
  prime     -> {'findings': {'prime': Fa

## What to notice

| Observation | Why |
|---|---|
| `validate` always runs first | it is the only node after 🏁 `START` |
| `low`, `mid`, `high` all run, even when two say "not mine" | 🔱 fan-out runs every branch; each decides for itself |
| Layer 3 starts only after all three range agents | 🔗 a list as the source of `add_edge` means "wait for all" |
| `finalize` runs last | it waits for all of layer 3 |
| Bad input skips layers 2 to 4 | the 🔀 conditional edge sent it to `reject` |
| `findings` holds keys from six different nodes | the 🧮 reducer merged the parallel writes |
| Each node returned only a few keys, yet `final` ends up in the state | LangGraph (here, our `apply`) merges every node's return into the 🟦 state |

## Where to look next in the project

| Concept | File |
|---|---|
| State, reducer, nodes, edges, parallel, conditional edges | `backend/src/triage/graph.py` |
| Agent pattern (read, announce, act, report) | `backend/src/triage/agents.py` |